# EMNIST: полная сетка v5 на Kaggle (T4 ×2 параллельно)

**Стратегия:** 2 эксперимента запускаются одновременно на двух T4 GPU. Когда один завершается — стартует следующий из очереди. Это сокращает общее время в **~2 раза**.

**Сетка:** v5 × 3 аугментации × 3 scheduler = **9 запусков**

**Параметры:** 25 эпох, seed=42, batch=64, lr=1e-3, AdamW + label_smoothing=0.1

**Ожидаемое время:** ~1 час на T4 ×2 (вместо ~2ч на одной).

**⚠️ Перед запуском:**
1. В Settings (правая панель) → `Accelerator` → `GPU T4 x2`
2. В `train.py` строка `device = torch.device("cuda" ...)` должна быть изменена на `device = torch.device("cuda:0" ...)`. Если не сделано — в самой первой ячейке есть автопатч.

## 1. Проверка GPU

In [ ]:
import torch
print(f'GPU доступен: {torch.cuda.is_available()}')
print(f'Количество GPU: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
!nvidia-smi

## 2. Клонирование репозитория

In [ ]:
REPO_URL = 'https://github.com/sultonovmuhibulloh612/emnist-master-thesis.git'

import os
os.chdir('/kaggle/working')

if not os.path.exists('/kaggle/working/emnist-project'):
    !git clone $REPO_URL /kaggle/working/emnist-project
%cd /kaggle/working/emnist-project

print('Текущая папка:', os.getcwd())

## 3. Установка зависимостей

In [ ]:
!pip install -q tensorboard seaborn

## 4. Автопатч train.py для совместимости с CUDA_VISIBLE_DEVICES

На случай, если в репо ещё не закоммичен патч — заменим `cuda` на `cuda:0` локально.

In [ ]:
with open('src/training/train.py', 'r', encoding='utf-8') as f:
    code = f.read()

old_line = 'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")'
new_line = 'device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")'

if old_line in code:
    code = code.replace(old_line, new_line)
    with open('src/training/train.py', 'w', encoding='utf-8') as f:
        f.write(code)
    print('✓ Патч применён: cuda → cuda:0')
elif new_line in code:
    print('✓ Патч уже применён ранее')
else:
    print('⚠ Не нашёл строку device = torch.device("cuda" ...)')
    print('  Проверь train.py вручную')

## 5. Проверка моделей и патчей

In [ ]:
v5_exists = os.path.exists('src/models/improved_cnn_v5.py')
print(f'{"✓" if v5_exists else "✗ ОТСУТСТВУЕТ"} src/models/improved_cnn_v5.py')

with open('src/training/train.py', encoding='utf-8') as f:
    code = f.read()

checks = {
    'AdamW в оптимизаторе':  'optim.AdamW' in code,
    'label_smoothing=0.1':    'label_smoothing=0.1' in code,
    'v5 в реестре моделей':   'improved_cnn_v5' in code,
    'cuda:0 (для параллели)': 'cuda:0' in code,
}
print()
for desc, ok in checks.items():
    print(f'{"✓" if ok else "✗"} {desc}')

## 6. Параллельный запуск 9 экспериментов на двух GPU

**Принцип работы:**
- Очередь из 9 экспериментов
- Запускаем по одному на каждую GPU (через `CUDA_VISIBLE_DEVICES`)
- Когда какой-то завершается — берём из очереди следующий
- Все 9 экспериментов проходят, используя обе T4 параллельно

In [ ]:
import subprocess, time, os
from datetime import datetime

MODEL = 'improved_cnn_v5'
AUGMENTATIONS = ['none', 'light', 'strong']
SCHEDULERS    = ['step', 'cosine', 'plateau']
SEED          = 42

EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, SPLIT = 25, 64, 1e-3, 1e-4, 'balanced'
NUM_GPUS = 2

# Очередь экспериментов
queue = [(aug, sch) for aug in AUGMENTATIONS for sch in SCHEDULERS]
total = len(queue)

def make_cmd(aug, sch, gpu_id):
    """Команда запуска одного эксперимента на указанной GPU."""
    exp_name = f'v5_{aug}_{sch}'
    cmd = [
        'python', '-m', 'src.training.train',
        '--model',        MODEL,
        '--experiment',   exp_name,
        '--epochs',       str(EPOCHS),
        '--batch_size',   str(BATCH_SIZE),
        '--lr',           str(LR),
        '--weight_decay', str(WEIGHT_DECAY),
        '--augmentation', aug,
        '--scheduler',    sch,
        '--split',        SPLIT,
        '--seed',         str(SEED),
        '--num_workers',  '2',
    ]
    # Логи в отдельные файлы, чтобы не мешались
    log_path = f'/kaggle/working/log_{exp_name}.txt'
    return cmd, exp_name, log_path

print(f'Всего экспериментов: {total}')
print(f'Параллельно на GPU: {NUM_GPUS}')
print(f'Старт: {datetime.now().strftime("%H:%M:%S")}')
print('=' * 70)

t_start = time.time()
results = []

# Слоты для GPU: [None, None] — обе свободны
slots = [None] * NUM_GPUS  # каждый элемент: None или (process, exp_name, gpu_id, t_start, log_path)
queue_idx = 0

# Главный цикл: пока есть что запускать или что-то крутится
while queue_idx < len(queue) or any(slot is not None for slot in slots):
    # 1. Запускаем новые эксперименты в свободные слоты
    for gpu_id in range(NUM_GPUS):
        if slots[gpu_id] is None and queue_idx < len(queue):
            aug, sch = queue[queue_idx]
            cmd, exp_name, log_path = make_cmd(aug, sch, gpu_id)

            # ВАЖНО: CUDA_VISIBLE_DEVICES изолирует процесс на нужную GPU
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)

            log_file = open(log_path, 'w')
            proc = subprocess.Popen(cmd, env=env, stdout=log_file, stderr=subprocess.STDOUT)

            t0 = time.time()
            slots[gpu_id] = (proc, exp_name, gpu_id, t0, log_file)
            queue_idx += 1

            print(f'[{queue_idx}/{total}] ▶ START GPU{gpu_id}: {exp_name}  '
                  f'({datetime.now().strftime("%H:%M:%S")})')

    # 2. Проверяем статус каждого слота
    for gpu_id in range(NUM_GPUS):
        if slots[gpu_id] is not None:
            proc, exp_name, _, t0, log_file = slots[gpu_id]
            ret = proc.poll()
            if ret is not None:
                # Процесс завершился
                elapsed = time.time() - t0
                log_file.close()
                status = 'OK' if ret == 0 else 'FAILED'
                marker = '✓' if ret == 0 else '✗'
                print(f'    {marker} DONE  GPU{gpu_id}: {exp_name}  '
                      f'({elapsed/60:.1f} мин, {datetime.now().strftime("%H:%M:%S")})')
                if ret != 0:
                    # Печатаем последние строки лога
                    with open(slots[gpu_id][4].name, 'r') as f:
                        tail = f.read().strip().split('\n')[-5:]
                    print('       Последние строки лога:')
                    for line in tail:
                        print(f'         {line}')

                results.append({
                    'experiment': exp_name,
                    'status':     status,
                    'time_min':   elapsed / 60,
                    'gpu_id':     gpu_id,
                })
                slots[gpu_id] = None

    # 3. Ждём чуть-чуть, чтобы не нагружать CPU опросом
    time.sleep(5)

print('\n' + '=' * 70)
total_min = (time.time() - t_start) / 60
print(f'ВСЕГО ВРЕМЕНИ: {total_min:.1f} мин ({total_min/60:.2f} ч)')
print(f'OK: {sum(1 for r in results if r["status"] == "OK")}/{total}')
print('\nСводка:')
for r in sorted(results, key=lambda x: x['experiment']):
    print(f"  {r['status']:6} | GPU{r['gpu_id']} | {r['time_min']:5.1f} мин | {r['experiment']}")

## 7. (Опционально) Посмотреть детальные логи отдельных экспериментов

Если какой-то эксперимент упал — здесь можно посмотреть полный stdout/stderr.

In [ ]:
# Список логов всех экспериментов
!ls -la /kaggle/working/log_*.txt

In [ ]:
# Посмотреть последние 30 строк конкретного лога — раскомментируй и подставь имя
# !tail -30 /kaggle/working/log_v5_strong_cosine.txt

## 8. Архивация результатов

### 8.1. Полный архив для тебя (со всеми файлами + hparams)

In [ ]:
# Полный архив — все файлы экспериментов v5 + папка hparams (если есть)
import os

tar_targets = ['results/*v5_none*', 'results/*v5_light*', 'results/*v5_strong*']
if os.path.exists('/kaggle/working/emnist-project/results/hparams'):
    tar_targets.append('results/hparams')
    print('✓ Папка hparams найдена — добавляю в архив')
else:
    print('⚠ Папки hparams нет — архивирую без неё')

tar_cmd = f'cd /kaggle/working/emnist-project && tar -czvf /kaggle/working/v5_grid_full.tar.gz {" ".join(tar_targets)} 2>&1 | tail -5'
!{tar_cmd}

!ls -lh /kaggle/working/v5_grid_full.tar.gz

### 8.2. Лёгкий архив для Claude (с exclude'ами)

In [ ]:
# Лёгкий архив — без .pth, без tensorboard
!cd /kaggle/working/emnist-project && \
  tar --exclude='*.pth' --exclude='*.pt' \
      --exclude='events.out.tfevents.*' --exclude='tensorboard' \
      -czvf /kaggle/working/v5_grid_light.tar.gz \
      results/*v5_none* results/*v5_light* results/*v5_strong* 2>&1 | tail -5

!ls -lh /kaggle/working/v5_grid_light.tar.gz

## 9. Скачивание архивов

В Kaggle файлы из `/kaggle/working/` доступны через панель **Output** справа.
После выполнения этой ячейки оба архива появятся в Output, откуда их можно скачать кнопкой Download.

In [ ]:
!ls -lh /kaggle/working/*.tar.gz
print('\n✓ Архивы готовы. Нажми "Output" в правой панели Kaggle и скачай файлы.')